# Screener Dashboard

Interactive Dash dashboard for exploring Magic Formula stock rankings. Loads the most recent `Results/stocks_YYYY-MM-DD.csv` automatically.

**Prerequisite:** a snapshot in `Results/` (the repo ships monthly ones; create a fresh one with `python -m magic_formula run`).

Snapshots hold the top 100 stocks by `Final_Rank`. Changing the ranking metrics below re-ranks those 100 rows only; the underlying rank values were computed against the full screened universe.

In [ ]:
import os
import glob
import pandas as pd
import dash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output

csv_path = None  # Set to a specific file path if you want to override the default behavior

In [ ]:
# Manual override for current date (for testing purposes)
# csv_path = './Results/stocks_2026-07-01.csv'

In [ ]:
# Auto-load the most recent Results CSV.
# To target a specific date, set: csv_path = './Results/stocks_YYYY-MM-DD.csv'
results_files = sorted(glob.glob('./Results/stocks_*.csv'))
if not results_files:
    raise FileNotFoundError(
        "No Results/stocks_*.csv found — run `python -m magic_formula run` first."
    )
if csv_path is None:
    csv_path = results_files[-1]
current_date = os.path.basename(csv_path).replace('stocks_', '').replace('.csv', '')
print(f"Loading: {csv_path}")

In [ ]:
# Load Data
df = pd.read_csv(f'./Results/stocks_{current_date}.csv')  # Specific file path for testing
#df = pd.read_csv(output_file_path) 
rank_cols = [col for col in df.columns if "_Rank" in col and col != "Combined_Rank" and col != "Final_Rank"]
info_cols = ["Company Name", "Ticker", "Market Cap", "Price"]

In [ ]:
# Initialize Dash App
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("Company Ranking Dashboard"),
    html.H2("Pulling data from: " + str(current_date)),
    html.Div([    
        html.Label("Select Ranking Metric(s):", className="filter-title", style={'fontWeight': 'bold', 'fontSize': '16px'}),
        dcc.Checklist(
            id='rank-selector',
            options=[{'label': col, 'value': col} for col in rank_cols],
            value=["MagicFormula_Rank", "PEGRatio_Rank", "PriceBook_Rank", 
                   "PriceFreeCashFlow_Rank", "PriceMomentum_20Day_Rank", "PriceMomentum_1Year_Rank"], # Default selection(s)
            inline=True
        ),
        html.Br(),
        html.Label("Individual Company Search:", className="filter-title", style={'fontWeight': 'bold', 'fontSize': '16px'}),
        dcc.Input(
            id='search-bar',
            type='text',
            placeholder='Search by Company Name or Ticker',
            style={'marginTop': '10px', 'width': '50%'}
        ),
        html.Br(),
        html.Label("Market Cap Range:", className="filter-title", style={'fontWeight': 'bold', 'fontSize': '16px', 'marginTop': '10px'}),
        html.Div([
            dcc.Input(id='min-market-cap', type='number', value=int(df['Market Cap'].min()), placeholder='Min Market Cap'),
            dcc.Input(id='max-market-cap', type='number', value=int(df['Market Cap'].max()), placeholder='Max Market Cap')
        ], style={'display': 'flex', 'gap': '50px'}),
        html.Br(),
        html.Label("Price Range:", className="filter-title", style={'fontWeight': 'bold', 'fontSize': '16px', 'marginTop': '10px'}),
        html.Div([
            dcc.Input(id='min-price', type='number', value=int(df['Price'].min()), placeholder='Min Price'),
            dcc.Input(id='max-price', type='number', value=int(df['Price'].max()), placeholder='Max Price')
        ], style={'display': 'flex', 'gap': '50px'}),
    ], className="filter-container"),
    
    html.Div([
        dash_table.DataTable(
            id='company-table',
            page_size=20,
            sort_action='native',
            style_table={'overflowX': 'auto'}
        )
    ], className="table-container"),
])

@app.callback(
    [Output('company-table', 'data'), Output('company-table', 'columns')],
    [Input('rank-selector', 'value')],
    [Input('search-bar', 'value')],
    [Input('min-market-cap', 'value'), Input('max-market-cap', 'value')],
    [Input('min-price', 'value'), Input('max-price', 'value')]
)
def update_table(selected_ranks, search_term, min_mc, max_mc, min_price, max_price):
    if not selected_ranks:
        return [], []  # Return empty dataset if no ranks selected
    filtered_df = df[info_cols + selected_ranks].copy()
    
    # Filter by Market Cap
    if min_mc is not None and max_mc is not None:
        filtered_df = filtered_df[(filtered_df['Market Cap'] >= min_mc) & (filtered_df['Market Cap'] <= max_mc)]
    
    # Filter by Price
    if min_price is not None and max_price is not None:
        filtered_df = filtered_df[(filtered_df['Price'] >= min_price) & (filtered_df['Price'] <= max_price)]
    
    filtered_df["Combined_Total"] = filtered_df[selected_ranks].sum(axis=1)
    filtered_df["Current_Rank"] = filtered_df["Combined_Total"].rank(method='first')
    filtered_df = filtered_df.sort_values("Combined_Total")
    columns = [{"name": col, "id": col} for col in filtered_df.columns]
    
    if search_term:
        search_term_lower = search_term.lower()
        filtered_df = filtered_df[
            filtered_df["Company Name"].str.lower().str.contains(search_term_lower) |
            filtered_df["Ticker"].str.lower().str.contains(search_term_lower)
        ]
    
    return filtered_df.to_dict('records'), columns


import os
if __name__ == '__main__' and not os.getenv('CI'):
    app.run(debug=True, port=8050)
